# Full training & evaluation pipeline

This notebook trains and evaluates **all 10 candidate models** end-to-end on the cached extractor features, then re-evaluates every one of them at a **strict decision threshold that holds FPR ≤ 1 %** (where the positive class is `ai`, so FP = a human text falsely flagged as AI).

Run cells top-to-bottom. Expected runtime: **~30–60 minutes on CPU** (neural training dominates), **~5–10 minutes on GPU**. All 10 trained checkpoints are saved to `models/ready_models/`.

## Architecture

Three feature extractors produce a (87 + 10 + 128 = 225)-dim feature triple per text. Two parallel training tracks consume the same cache:

```
                  ┌─ NELA          (87)  surface / linguistic features
  text  ────────► ├─ StyleDecipher (10)  similarity vs 5 LLM rewrites (mean + std of 5 metrics)
                  └─ TRACE         (128) contrastive author-style embedding
                       │
                       ├──► Neural fusion track   (4 strategies)  → 4 .pt   checkpoints
                       │      concat | mlp | attention | gating
                       │      MultiFeatureFusion → 2-class softmax head
                       │
                       └──► Classical track       (6 backends)    → 6 .joblib checkpoints
                              xgboost | random_forest | logreg | svm | hist_gbm | gradient_boosting
                              feature triple flattened to 225-dim → model fits directly
```

The **10 models** we'll train and compare:

| Track | Models |
|---|---|
| Neural fusion | `concat`, `mlp`, `attention`, `gating` |
| Classical | `xgboost`, `random_forest`, `logreg`, `svm`, `hist_gbm`, `gradient_boosting` |

Classical tree/linear models additionally report **per-extractor feature importance** — answering the core question of this repo: *how much does each extractor contribute?*

## 1. Setup — imports, paths, sanity check

In [12]:
import sys, json
from pathlib import Path

# Make sure the repo root is on sys.path so the training package imports work
NB_DIR = Path.cwd() if Path.cwd().name == "training" else Path.cwd() / "training"
REPO_ROOT = NB_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    confusion_matrix, roc_curve, roc_auc_score,
)

from training import paths
from training.config import TrainConfig, FUSION_METHODS
from training.feature_dataset import FeatureNormalizer, FusionFeatureDataset
from training.classical import BACKENDS, ClassicalClassifier, flatten_features
from training.model import FusionClassifier
from training.train import train_one as train_neural_one
from training.train_classical import train_one as train_classical_one

FEATURE_DIR = paths.FEATURE_DIR
SAVE_DIR    = paths.READY_MODELS_DIR
SAVE_DIR.mkdir(parents=True, exist_ok=True)

for sp in ("train", "val", "test"):
    assert (FEATURE_DIR / f"{sp}.npz").exists(), (
        f"missing {sp}.npz — run `python -m training.build_dataset --splits all --styledecipher ollama` first"
    )
print("feature cache OK — splits found:", sorted(p.name for p in FEATURE_DIR.glob("*.npz")))
print("save dir:", SAVE_DIR)

feature cache OK — splits found: ['test.npz', 'train.npz', 'val.npz']
save dir: C:\Users\Dimin\Documents\Hanyang\Smart Mobile Computing (AI-driven approaches)\repos\modern-AI-detection-trends-comparison\models\ready_models


## 2. Load features and fit the normalizer

Both tracks share one `FeatureNormalizer` fit on the **un-normalized** training split. The normalizer's mean/std are saved into every checkpoint so evaluation later applies the same transform.

In [13]:
train_ds = FusionFeatureDataset(FEATURE_DIR / "train.npz")
val_ds   = FusionFeatureDataset(FEATURE_DIR / "val.npz")
test_ds_raw = FusionFeatureDataset(FEATURE_DIR / "test.npz")  # un-normalized — for evaluation later

normalizer = FeatureNormalizer.fit(train_ds.raw_arrays())
train_ds.apply_normalizer(normalizer)
val_ds.apply_normalizer(normalizer)

dims = train_ds.dims
print(f"dims                 = {dims}")
print(f"train / val / test   = {len(train_ds)} / {len(val_ds)} / {len(test_ds_raw)}")
print(f"train class counts   = {train_ds.class_counts()}")
print(f"style coverage       = train {train_ds.style_coverage():.1%}, val {val_ds.style_coverage():.1%}, test {test_ds_raw.style_coverage():.1%}")

dims                 = {'nela': 87, 'style': 10, 'trace': 128}
train / val / test   = 9265 / 1846 / 1975
train class counts   = {0: 2406, 1: 6859}
style coverage       = train 100.0%, val 100.0%, test 100.0%


## 3. Train the 4 neural fusion models

Each `train_one` call runs up to 40 epochs of `MultiFeatureFusion` with the given strategy, keeps the best-val-F1 checkpoint, and writes `<name>.pt` + `<name>.metrics.json` to `models/ready_models/`.

In [14]:
neural_paths = {}
for method in FUSION_METHODS:
    print(f"\n{'=' * 60}\n   neural fusion: {method}\n{'=' * 60}")
    config = TrainConfig(fusion_method=method)
    out_path = SAVE_DIR / f"fusion_{method}.pt"
    train_neural_one(config, train_ds, val_ds, normalizer, out_path)
    neural_paths[method] = out_path


   neural fusion: concat

[concat] device=cuda  train=9265 val=1846  class_counts={0: 2406, 1: 6859}
  epoch   1/40  train_loss=0.2031 acc=0.910  |  val_loss=0.1373 acc=0.946 macroF1=0.9328  <- best
  epoch   2/40  train_loss=0.0861 acc=0.967  |  val_loss=0.0967 acc=0.967 macroF1=0.9585  <- best
  epoch   3/40  train_loss=0.0613 acc=0.975  |  val_loss=0.1082 acc=0.963 macroF1=0.9530
  epoch   4/40  train_loss=0.0426 acc=0.985  |  val_loss=0.1040 acc=0.967 macroF1=0.9584
  epoch   5/40  train_loss=0.0329 acc=0.988  |  val_loss=0.1047 acc=0.968 macroF1=0.9593  <- best
  epoch   6/40  train_loss=0.0233 acc=0.992  |  val_loss=0.1378 acc=0.965 macroF1=0.9554
  epoch   7/40  train_loss=0.0160 acc=0.995  |  val_loss=0.1611 acc=0.969 macroF1=0.9600  <- best
  epoch   8/40  train_loss=0.0127 acc=0.995  |  val_loss=0.1965 acc=0.962 macroF1=0.9508
  epoch   9/40  train_loss=0.0300 acc=0.989  |  val_loss=0.1188 acc=0.964 macroF1=0.9539
  epoch  10/40  train_loss=0.0135 acc=0.995  |  val_loss=0.16

## 4. Train the 6 classical classifiers

These flatten the three feature blocks into one 225-dim vector and let the classifier do its own combination. Tree/linear models also report per-extractor importance, surfacing how much each of NELA / StyleDecipher / TRACE drove the prediction.

`xgboost` is skipped automatically if not installed; everything else uses scikit-learn.

In [15]:
X_train = flatten_features(train_ds.nela, train_ds.style, train_ds.trace)
y_train = train_ds.labels
X_val   = flatten_features(val_ds.nela,   val_ds.style,   val_ds.trace)
y_val   = val_ds.labels
print(f"flattened feature width: {X_train.shape[1]}  (= {dims['nela']} + {dims['style']} + {dims['trace']})")

available = ClassicalClassifier.available_backends()
skipped = [b for b in BACKENDS if b not in available]
if skipped:
    print(f"skipping unavailable backends: {skipped}")

classical_paths = {}
overrides = {"n_estimators": None, "max_depth": None, "learning_rate": None}
for backend in available:
    print(f"\n{'=' * 60}\n   classical: {backend}\n{'=' * 60}")
    out_path = SAVE_DIR / f"clf_{backend}.joblib"
    train_classical_one(
        backend, X_train, y_train, X_val, y_val, dims, normalizer, out_path,
        seed=42, class_weighting=True, overrides=overrides,
    )
    classical_paths[backend] = out_path

flattened feature width: 225  (= 87 + 10 + 128)

   classical: xgboost

[xgboost] fitting on 9265 records ...
  extractor importance:  nela=69.4%  style=7.5%  trace=23.1%
  val acc=0.9724  macroF1=0.9647  (10.7s)  -> saved C:\Users\Dimin\Documents\Hanyang\Smart Mobile Computing (AI-driven approaches)\repos\modern-AI-detection-trends-comparison\models\ready_models\clf_xgboost.joblib

   classical: random_forest

[random_forest] fitting on 9265 records ...
  extractor importance:  nela=63.9%  style=10.8%  trace=25.3%
  val acc=0.9345  macroF1=0.9109  (5.5s)  -> saved C:\Users\Dimin\Documents\Hanyang\Smart Mobile Computing (AI-driven approaches)\repos\modern-AI-detection-trends-comparison\models\ready_models\clf_random_forest.joblib

   classical: logreg

[logreg] fitting on 9265 records ...
  extractor importance:  nela=44.5%  style=17.7%  trace=37.8%
  val acc=0.9209  macroF1=0.9018  (0.3s)  -> saved C:\Users\Dimin\Documents\Hanyang\Smart Mobile Computing (AI-driven approaches)\repos\mo

## 5. Evaluate all 10 on the held-out test split — default threshold (0.5)

Each saved checkpoint carries its own copy of the normalizer, so we reload the **un-normalized** test split per evaluation, apply the checkpoint's normalizer, then predict. Identical logic to [`test/evaluate.py`](../test/evaluate.py).

In [16]:
import torch

TEST_NPZ = FEATURE_DIR / "test.npz"

def predict(model_path):
    """Return (preds, ai_probs, labels) for either a .pt or .joblib checkpoint."""
    ds = FusionFeatureDataset(TEST_NPZ)

    if model_path.suffix == ".pt":
        model, payload = FusionClassifier.load(model_path, map_location="cpu")
        norm = FeatureNormalizer.from_state_dict(payload.get("normalizer"))
        if norm is not None:
            ds.apply_normalizer(norm)
        model.eval()
        with torch.no_grad():
            logits = model(
                torch.from_numpy(ds.nela),
                torch.from_numpy(ds.style),
                torch.from_numpy(ds.trace),
            )
            probs = torch.softmax(logits, dim=-1)[:, 1].numpy()
            preds = logits.argmax(dim=-1).numpy()
    else:
        clf, payload = ClassicalClassifier.load(model_path)
        norm = FeatureNormalizer.from_state_dict(payload.get("normalizer"))
        if norm is not None:
            ds.apply_normalizer(norm)
        X = flatten_features(ds.nela, ds.style, ds.trace)
        probs = clf.predict_proba(X)[:, 1]
        preds = clf.predict(X)

    return preds.astype(int), probs.astype(float), ds.labels.astype(int)

all_models = {f"neural_{m}": p for m, p in neural_paths.items()}
all_models.update({f"classical_{b}": p for b, p in classical_paths.items()})
print("models to evaluate:")
for k, v in all_models.items():
    print(f"  {k:<32} {v.name}")

models to evaluate:
  neural_concat                    fusion_concat.pt
  neural_mlp                       fusion_mlp.pt
  neural_attention                 fusion_attention.pt
  neural_gating                    fusion_gating.pt
  classical_xgboost                clf_xgboost.joblib
  classical_random_forest          clf_random_forest.joblib
  classical_logreg                 clf_logreg.joblib
  classical_svm                    clf_svm.joblib
  classical_mlp                    clf_mlp.joblib
  classical_hist_gbm               clf_hist_gbm.joblib
  classical_gradient_boosting      clf_gradient_boosting.joblib


In [17]:
# Run prediction for every model once; reuse probs/preds in subsequent cells.
predictions = {}
for name, path in all_models.items():
    preds, probs, labels = predict(path)
    predictions[name] = {"preds": preds, "probs": probs, "labels": labels}
    acc = accuracy_score(labels, preds)
    mF1 = f1_score(labels, preds, average="macro", zero_division=0)
    auc = roc_auc_score(labels, probs)
    print(f"  {name:<32} acc={acc:.4f}  macroF1={mF1:.4f}  ROC-AUC={auc:.4f}")

  neural_concat                    acc=0.9792  macroF1=0.9730  ROC-AUC=0.9971
  neural_mlp                       acc=0.9792  macroF1=0.9730  ROC-AUC=0.9971
  neural_attention                 acc=0.9701  macroF1=0.9611  ROC-AUC=0.9946
  neural_gating                    acc=0.9767  macroF1=0.9699  ROC-AUC=0.9964
  classical_xgboost                acc=0.9732  macroF1=0.9650  ROC-AUC=0.9966
  classical_random_forest          acc=0.9347  macroF1=0.9094  ROC-AUC=0.9871
  classical_logreg                 acc=0.9119  macroF1=0.8917  ROC-AUC=0.9696
  classical_svm                    acc=0.9590  macroF1=0.9481  ROC-AUC=0.9931
  classical_mlp                    acc=0.9630  macroF1=0.9523  ROC-AUC=0.9933
  classical_hist_gbm               acc=0.9757  macroF1=0.9683  ROC-AUC=0.9967
  classical_gradient_boosting      acc=0.9534  macroF1=0.9407  ROC-AUC=0.9921


In [18]:
# Confusion matrix per model at default threshold (=argmax / 0.5).
# Layout: rows = true class, cols = predicted class. Positive class (1) = AI.

def fmt_cm(cm, label_names=("human", "ai")):
    out = ["            " + "".join(f"{n:>8}" for n in label_names)]
    for i, n in enumerate(label_names):
        out.append(f"  true {n:<5}" + "".join(f"{int(v):>8}" for v in cm[i]))
    return "\n".join(out)

for name in all_models:
    p = predictions[name]
    cm = confusion_matrix(p["labels"], p["preds"], labels=[0, 1])
    print(f"\n=== {name} ===")
    print(fmt_cm(cm))


=== neural_concat ===
               human      ai
  true human     492      25
  true ai         16    1442

=== neural_mlp ===
               human      ai
  true human     492      25
  true ai         16    1442

=== neural_attention ===
               human      ai
  true human     483      34
  true ai         25    1433

=== neural_gating ===
               human      ai
  true human     495      22
  true ai         24    1434

=== classical_xgboost ===
               human      ai
  true human     485      32
  true ai         21    1437

=== classical_random_forest ===
               human      ai
  true human     401     116
  true ai         13    1445

=== classical_logreg ===
               human      ai
  true human     474      43
  true ai        131    1327

=== classical_svm ===
               human      ai
  true human     494      23
  true ai         58    1400

=== classical_mlp ===
               human      ai
  true human     482      35
  true ai         38  

## 6. Threshold tuning — enforce **FPR ≤ 1 %** (humans falsely flagged as AI)

Positive class = AI ⇒ a **false positive is a human text predicted as AI**. The default 0.5 threshold optimises accuracy but admits a relatively high FPR. To deploy the detector for screening (where wrongly accusing humans is much worse than missing an AI), we want **FPR ≤ 1 %** while keeping AI recall (TPR) as high as possible.

**Methodology** (avoids test-set leakage):
1. For each model, scan thresholds on the **validation** AI-probabilities and pick the *lowest* threshold whose val-FPR ≤ 0.01 (= highest TPR subject to the constraint).
2. Apply that threshold to the **test** probabilities and report the actual FPR / TPR / precision / confusion matrix it produces there.

If a model's val FPR can't reach 0.01 at any threshold (e.g. it's overconfident on humans), we mark it infeasible.

In [19]:
# Compute val probs once per model (parallel structure to predict() but for val split).
VAL_NPZ = FEATURE_DIR / "val.npz"

def predict_val(model_path):
    ds = FusionFeatureDataset(VAL_NPZ)
    if model_path.suffix == ".pt":
        model, payload = FusionClassifier.load(model_path, map_location="cpu")
        norm = FeatureNormalizer.from_state_dict(payload.get("normalizer"))
        if norm is not None:
            ds.apply_normalizer(norm)
        model.eval()
        with torch.no_grad():
            logits = model(
                torch.from_numpy(ds.nela),
                torch.from_numpy(ds.style),
                torch.from_numpy(ds.trace),
            )
            probs = torch.softmax(logits, dim=-1)[:, 1].numpy()
    else:
        clf, payload = ClassicalClassifier.load(model_path)
        norm = FeatureNormalizer.from_state_dict(payload.get("normalizer"))
        if norm is not None:
            ds.apply_normalizer(norm)
        X = flatten_features(ds.nela, ds.style, ds.trace)
        probs = clf.predict_proba(X)[:, 1]
    return probs.astype(float), ds.labels.astype(int)


def pick_threshold(val_probs, val_labels, max_fpr=0.01):
    """Highest TPR threshold whose FPR on val is ≤ max_fpr."""
    fpr_arr, tpr_arr, thr_arr = roc_curve(val_labels, val_probs)
    feasible = fpr_arr <= max_fpr
    if not feasible.any():
        return None, None, None
    # roc_curve sorts thresholds descending → fpr/tpr ascending. We want the LAST
    # index where fpr ≤ max_fpr (largest tpr that still respects the constraint).
    idx = int(np.where(feasible)[0][-1])
    return float(thr_arr[idx]), float(fpr_arr[idx]), float(tpr_arr[idx])


def metrics_at_threshold(probs, labels, threshold):
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    acc = (tp + tn) / cm.sum()
    return {"fpr": fpr, "tpr": tpr, "precision": prec, "accuracy": acc, "cm": cm}

In [20]:
MAX_FPR = 0.01

strict_results = {}
for name, path in all_models.items():
    val_probs, val_labels = predict_val(path)
    thr, val_fpr, val_tpr = pick_threshold(val_probs, val_labels, max_fpr=MAX_FPR)

    if thr is None:
        print(f"\n=== {name} ===  INFEASIBLE — no threshold achieves val FPR ≤ {MAX_FPR:.0%}")
        strict_results[name] = None
        continue

    test_p = predictions[name]
    test_metrics = metrics_at_threshold(test_p["probs"], test_p["labels"], thr)
    strict_results[name] = {"threshold": thr, "val_fpr": val_fpr, "val_tpr": val_tpr, **test_metrics}

    print(f"\n=== {name} ===")
    print(f"  threshold (picked on val)  : {thr:.4f}")
    print(f"  val   FPR / TPR @ threshold: {val_fpr:.4f} / {val_tpr:.4f}")
    print(f"  test  FPR / TPR @ threshold: {test_metrics['fpr']:.4f} / {test_metrics['tpr']:.4f}")
    print(f"  test  precision / accuracy : {test_metrics['precision']:.4f} / {test_metrics['accuracy']:.4f}")
    print("  test confusion matrix:")
    print(fmt_cm(test_metrics["cm"]))


=== neural_concat ===
  threshold (picked on val)  : 0.9996
  val   FPR / TPR @ threshold: 0.0081 / 0.8885
  test  FPR / TPR @ threshold: 0.0000 / 0.8669
  test  precision / accuracy : 1.0000 / 0.9018
  test confusion matrix:
               human      ai
  true human     517       0
  true ai        194    1264

=== neural_mlp ===
  threshold (picked on val)  : 0.9996
  val   FPR / TPR @ threshold: 0.0081 / 0.8885
  test  FPR / TPR @ threshold: 0.0000 / 0.8669
  test  precision / accuracy : 1.0000 / 0.9018
  test confusion matrix:
               human      ai
  true human     517       0
  true ai        194    1264

=== neural_attention ===
  threshold (picked on val)  : 0.9992
  val   FPR / TPR @ threshold: 0.0081 / 0.8833
  test  FPR / TPR @ threshold: 0.0039 / 0.8841
  test  precision / accuracy : 0.9985 / 0.9134
  test confusion matrix:
               human      ai
  true human     515       2
  true ai        169    1289

=== neural_gating ===
  threshold (picked on val)  : 0.99

## 7. Side-by-side comparison: default 0.5 vs FPR ≤ 1 %

The two regimes answer different questions:
* **default (argmax)** — what's the best balanced accuracy?
* **FPR ≤ 1 %** — if I refuse to wrongly accuse more than 1 % of humans, what fraction of AI text can I still catch?

In [21]:
import pandas as pd

rows = []
for name in all_models:
    p = predictions[name]
    cm_def = confusion_matrix(p["labels"], p["preds"], labels=[0, 1])
    tn, fp, fn, tp = cm_def.ravel()
    fpr_def = fp / (fp + tn) if (fp + tn) else 0.0
    tpr_def = tp / (tp + fn) if (tp + fn) else 0.0

    sr = strict_results.get(name)
    row = {
        "model":           name,
        "acc@0.5":         accuracy_score(p["labels"], p["preds"]),
        "macroF1@0.5":     f1_score(p["labels"], p["preds"], average="macro", zero_division=0),
        "ROC-AUC":         roc_auc_score(p["labels"], p["probs"]),
        "FPR@0.5":         fpr_def,
        "TPR@0.5":         tpr_def,
        "threshold@FPR≤1%":sr["threshold"] if sr else float("nan"),
        "FPR@strict":      sr["fpr"]       if sr else float("nan"),
        "TPR@strict":      sr["tpr"]       if sr else float("nan"),
        "precision@strict":sr["precision"] if sr else float("nan"),
    }
    rows.append(row)

df = pd.DataFrame(rows).sort_values("TPR@strict", ascending=False).reset_index(drop=True)
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
df

,model,acc@0.5,macroF1@0.5,ROC-AUC,FPR@0.5,TPR@0.5,threshold@FPR≤1%,FPR@strict,TPR@strict,precision@strict
0,classical_hist_gbm,0.9757,0.9683,0.9967,0.0580,0.9877,0.9870,0.0097,0.9438,0.9964
1,classical_xgboost,0.9732,0.9650,0.9966,0.0619,0.9856,0.9535,0.0116,0.9273,0.9956
2,neural_attention,0.9701,0.9611,0.9946,0.0658,0.9829,0.9992,0.0039,0.8841,0.9985
3,classical_gradient_boosting,0.9534,0.9407,0.9921,0.0638,0.9595,0.8609,0.0135,0.8841,0.9946
4,classical_svm,0.9590,0.9481,0.9931,0.0445,0.9602,0.9451,0.0058,0.8745,0.9977
5,classical_mlp,0.9630,0.9523,0.9933,0.0677,0.9739,0.9972,0.0058,0.8697,0.9976
6,neural_gating,0.9767,0.9699,0.9964,0.0426,0.9835,0.9998,0.0019,0.8683,0.9992
7,neural_concat,0.9792,0.9730,0.9971,0.0484,0.9890,0.9996,0.0000,0.8669,1.0000
8,neural_mlp,0.9792,0.9730,0.9971,0.0484,0.9890,0.9996,0.0000,0.8669,1.0000
9,classical_random_forest,0.9347,0.9094,0.9871,0.2244,0.9911,0.7925,0.0097,0.8176,0.9958


## 8. Persist the comparison + verify saved checkpoints

Writes `models/ready_models/pipeline_summary.json` next to the checkpoints so the comparison survives the notebook session and can be diffed across re-runs.

In [22]:
summary = {
    "max_fpr_constraint": MAX_FPR,
    "models": [],
}
for _, r in df.iterrows():
    summary["models"].append({k: (None if (isinstance(v, float) and np.isnan(v)) else v) for k, v in r.items()})

summary_path = SAVE_DIR / "pipeline_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))
print(f"wrote {summary_path}\n")

print("saved checkpoints in", SAVE_DIR)
for p in sorted(SAVE_DIR.iterdir()):
    if p.suffix in (".pt", ".joblib"):
        print(f"  {p.name:<32} {p.stat().st_size/1024:>10.1f} KB")
    elif p.suffix == ".json":
        print(f"  {p.name:<32} {p.stat().st_size/1024:>10.1f} KB  (metrics / summary)")

wrote C:\Users\Dimin\Documents\Hanyang\Smart Mobile Computing (AI-driven approaches)\repos\modern-AI-detection-trends-comparison\models\ready_models\pipeline_summary.json

saved checkpoints in C:\Users\Dimin\Documents\Hanyang\Smart Mobile Computing (AI-driven approaches)\repos\modern-AI-detection-trends-comparison\models\ready_models
  clf_gradient_boosting.joblib          541.0 KB
  clf_gradient_boosting.metrics.json        0.6 KB  (metrics / summary)
  clf_hist_gbm.joblib                  1851.9 KB
  clf_hist_gbm.metrics.json               0.4 KB  (metrics / summary)
  clf_logreg.joblib                       7.1 KB
  clf_logreg.metrics.json                 0.6 KB  (metrics / summary)
  clf_mlp.joblib                       1075.3 KB
  clf_mlp.metrics.json                    0.4 KB  (metrics / summary)
  clf_random_forest.joblib            34004.8 KB
  clf_random_forest.metrics.json          0.6 KB  (metrics / summary)
  clf_svm.joblib                       5428.7 KB
  clf_svm.metrics.

## 9. Reading the results

* **`acc@0.5`, `macroF1@0.5`** — standard headline numbers from `argmax` predictions.
* **`ROC-AUC`** — threshold-free ranking quality. A high AUC with a low `TPR@strict` means the model ranks samples correctly but its probabilities aren't sharp enough for a strict cutoff.
* **`FPR@strict`** — measured on the test set; should be close to (but may slightly exceed) 0.01 because the threshold was picked on val.
* **`TPR@strict`** — the key deployment number: "if I'm allowed to wrongly accuse ≤ 1 % of humans, what fraction of AI text do I catch?".
* **`precision@strict`** — among everything the model flags as AI under the strict regime, what fraction really is AI.

All 10 checkpoints are now in [`../models/ready_models/`](../models/ready_models/) and can be loaded later with:

```bash
python -m test.evaluate --model models/ready_models/fusion_gating.pt
python -m test.evaluate --model models/ready_models/clf_xgboost.joblib
```

See [`README.md`](README.md) for the training-framework documentation and [`../README.md`](../README.md) for the project overview.